# 14 — Model Building for Clustering (Objective 1)

**Objective.** Build the two clustering models used for Objective 1 — **K-Means** and **hierarchical clustering** — on
the four standardised performance measures, choose the number of segments on evidence, and give every one of the
25,000 warehouses a segment.

For clustering, *hyperparameter tuning* means above all choosing **k**, the number of clusters. There is no correct
answer to check against, so k is chosen from several independent kinds of evidence rather than one score.

Building on what the EDA and feature engineering steps established: the performance data is a continuum, so separation
scores are expected to be *modest*. The 908 unrated warehouses are assigned to a separate group by rule; the models
run on the 24,092 rated warehouses.

**Input:** `Obj_1_Clustering/data/processed/clustering_input_scaled.csv`
**Output:** `model/kmeans.pkl` · `training_and_evaluation/k_selection.csv` · `training_and_evaluation/segment_labels.csv`
· `training_and_evaluation/ward_vs_kmeans_sample.csv` · figures in `training_and_evaluation/`

## 0. Setup

In [ ]:
import sys, pathlib

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "common.py").exists())
sys.path.insert(0, str(ROOT))
from src.common import *

set_style()

paths = obj_paths(1)
scaled = pd.read_csv(paths["processed"] / "clustering_input_scaled.csv")
inputs = ["product_wg_ton", "num_refill_req_l3m", "transport_issue_l1y", "wh_breakdown_l3m"]
X = scaled[inputs].to_numpy()

print(f"clustering input: {X.shape[0]:,} warehouses x {X.shape[1]} standardised measures")

In [ ]:
from itertools import combinations
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score, adjusted_rand_score)
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram

---
## 1. Scanning k from 2 to 10

K-Means is fitted once for each k, and four measures are recorded. Each looks at the result differently:

| Measure | What it asks | Better when |
|---|---|---|
| **Inertia** | total squared distance from each warehouse to its cluster centre | it stops falling steeply as k rises — the *elbow* |
| **Silhouette** | is each warehouse closer to its own cluster than to the next nearest? | higher (−1 to 1) |
| **Davies–Bouldin** | how spread out clusters are relative to how far apart they are | lower |
| **Calinski–Harabasz** | variance between clusters relative to variance within them | higher |

Two further columns: **`smallest_cluster_pct`** — a segment holding a tiny share of warehouses is hard to act on — and
**`inertia_drop_pct`**, the percentage by which inertia falls compared with k − 1, which makes the elbow readable as
numbers rather than only by eye.

A common rule of thumb for the silhouette score reads above 0.70 as strong structure, 0.51–0.70 as reasonable,
0.26–0.50 as weak, and 0.25 or below as no substantial structure.

**Settings.** `n_init=10` runs K-Means from ten different starting points and keeps the best, because a single start
can settle on a poor solution; §2 checks whether ten is enough. `random_state` is fixed so the results reproduce.

**The silhouette is computed on a sample.** It compares every warehouse with every other, and the cell prints how many
pairs that is per k. A fixed sample of 10,000 warehouses — the same sample for every k — keeps the comparison between
values of k fair.

In [ ]:
n = len(X)
print(f"warehouse pairs a full silhouette would compare, per k: {n * (n - 1) // 2:,}\n")

ks = list(range(2, 11))
rows = []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    sizes = np.bincount(km.labels_)
    rows.append({
        "k": k,
        "inertia": round(km.inertia_, 1),
        "silhouette": round(silhouette_score(X, km.labels_, sample_size=10_000, random_state=RANDOM_STATE), 4),
        "davies_bouldin": round(davies_bouldin_score(X, km.labels_), 4),
        "calinski_harabasz": round(calinski_harabasz_score(X, km.labels_), 1),
        "smallest_cluster_pct": round(100 * sizes.min() / n, 2),
    })

scan = pd.DataFrame(rows).set_index("k")
scan.insert(1, "inertia_drop_pct", (-100 * scan["inertia"].pct_change()).round(2))
scan

> **Interpretation.**
>
> Reading the table before the plots: inertia drops steeply from k = 2 to k = 4 — by 20.70% and 16.96% — then
> by only 11.54% to 11.73% for k = 5 and 6, and under 8% from k = 7 onward. The steepest slowdown in those
> percentage drops comes immediately after k = 4. Silhouette scores range from 0.2204 to 0.2437 across all nine
> values of k — all at or below the 0.25 threshold the rule of thumb associates with no substantial structure,
> consistent with the continuum the earlier analysis found. Calinski–Harabasz peaks at k = 4 (7,661.0) and falls
> on either side. Davies–Bouldin keeps improving until k = 6 (1.1725) and then wavers. No segment falls below
> about 6% of the warehouses until k = 10. The four-panel plot below visualises these patterns.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
panels = [("inertia", "Inertia — look for the elbow"),
          ("silhouette", "Silhouette — higher is better"),
          ("davies_bouldin", "Davies–Bouldin — lower is better"),
          ("calinski_harabasz", "Calinski–Harabasz — higher is better")]
for ax, (col, title) in zip(axes.ravel(), panels):
    ax.plot(scan.index, scan[col], marker="o")
    ax.set_title(title); ax.set_xlabel("k"); ax.set_xticks(ks)
fig.suptitle("Choosing k — K-Means on 24,092 rated warehouses", fontweight="bold")
plt.tight_layout()
save_fig(paths["train_eval"] / "k_selection.png")
plt.show()

> **Interpretation.**
>
> - **Silhouette confirms the expectation that the data is a continuum.** Every k scores between 0.2204 (k = 2)
>   and 0.2437 (k = 10) — all at or below 0.25, the band the rule of thumb reads as no substantial structure.
>   The plot's y-axis spans only about 0.02, so its zig-zag exaggerates differences that are negligible in
>   practice. Silhouette cannot choose k here. This is what dividing a continuum looks like, not a failed model.
>
> - **Calinski–Harabasz peaks clearly at k = 4** (7,661.0), falling on either side (7,499.0 at k = 3, 7,281.2 at k = 5).
>
> - **Davies–Bouldin keeps improving until k = 6** (1.2976 at 4, 1.2173 at 5, 1.1725 at 6), then wavers — rising to
>   1.2758 at k = 8 before reaching its lowest, 1.1701, at k = 10.
>
> - **Inertia has no sharp elbow.** Read from the plot, the curve bends gently. In numbers, each added cluster cuts
>   inertia by 20.70% (k = 3) and 16.96% (k = 4), then by only 11.54% (k = 5) and 11.73% (k = 6), and by less than 8%
>   from k = 7 onward. The biggest fall in those percentages comes immediately after k = 4.
>
> - **No segment becomes unworkably small until k = 8.** The smallest cluster holds 16.68% of warehouses at k = 4 and
>   14.14% at k = 6, but 8.37% at k = 8 and 6.02% at k = 10.
>
> - The measures do not all agree: Calinski–Harabasz and the elbow point to 4; Davies–Bouldin to 6 or more. §2 adds
>   evidence that separates them.

---
## 2. Stability — does K-Means find the same segments each time?

NB 11 §6 found no natural groups. When data has no natural groups, there can be many different ways to divide it that
are almost equally good, and K-Means may land on a different one depending on where it starts. A segmentation that
changes from run to run cannot be described to a manager as "the" segments.

For each k, K-Means is refitted with **five different random seeds** (each still using 10 starts), and every pair of
runs is compared with the **Adjusted Rand Index (ARI)**. ARI measures how far two labellings agree on which warehouses
belong together, regardless of the numbers attached to the clusters: **1** means identical segments, **0** means no
more agreement than chance.

The same cell checks the `n_init` setting: if raising it from 10 to 50 does not lower inertia, ten starts are enough.

In [ ]:
rows = []
for k in ks:
    runs = [KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE + s).fit(X) for s in range(1, 6)]
    aris = [adjusted_rand_score(a.labels_, b.labels_) for a, b in combinations(runs, 2)]
    inertia_50 = KMeans(n_clusters=k, n_init=50, random_state=RANDOM_STATE).fit(X).inertia_
    rows.append({
        "k": k,
        "mean_ari_between_runs": round(np.mean(aris), 4),
        "lowest_ari": round(np.min(aris), 4),
        "inertia_n_init_10": scan.loc[k, "inertia"],
        "inertia_n_init_50": round(inertia_50, 1),
    })

stability = pd.DataFrame(rows).set_index("k")
stability["n_init_50_improves_by_pct"] = (100 * (stability["inertia_n_init_10"] - stability["inertia_n_init_50"])
                                          / stability["inertia_n_init_10"]).round(4)
scan = scan.join(stability[["mean_ari_between_runs", "lowest_ari"]])
stability

> **Interpretation.**
>
> - Stability separates the candidates sharply.
>
> - **k = 4 is the most stable solution of all.** Five differently seeded runs agree at a mean ARI of **0.9997**, and
>   even the least similar pair at 0.9992. K-Means finds the same four segments every time. k = 3 is nearly as stable
>   (0.9986).
> - **k = 5 is unstable.** Mean ARI 0.7528, with the least similar pair at just **0.4298** — two runs can divide the
>   warehouses quite differently. With five clusters there are several near-equal ways to cut the continuum, and the
>   starting point decides which one K-Means settles on.
> - **k = 6**, Davies–Bouldin's preference, is stable but less so (0.9639; lowest 0.9391), and stability keeps
>   declining toward k = 10 (0.8520).
> - **Ten starts are enough.** Raising `n_init` from 10 to 50 lowers inertia by at most 0.0236% at any k, and by
>   0.0000% at k = 4: the extra starts find nothing better.


---
## 3. The number of segments

> **Decision — segment count.**
>
> - Use **k = 4** for the 24,092 rated warehouses.
> - Add **segment 0** for the 908 unrated warehouses by rule.
> - Report **five segments in all**.
>
> | Evidence | k = 4 | k = 6 (Davies–Bouldin's preference) |
> |---|---|---|
> | Calinski–Harabasz | **7,661.0 — the maximum** | 7,238.5 |
> | Stability, mean ARI | **0.9997 — the most stable** | 0.9639 |
> | Inertia cut from the previous k | 16.96%, before cuts shrink to 11.54% | 11.73% |
> | Davies–Bouldin | 1.2976 | **1.1725** |
> | Silhouette | 0.2403 | 0.2376 |
> | Smallest segment | 16.68% | 14.14% |
>
> - Calinski–Harabasz favours k = 4.
> - Stability favours k = 4.
> - The elbow favours k = 4.
> - Silhouette is indifferent across the tested k values.
> - Davies–Bouldin alone favours k = 6.
> - Reject k = 3: stable, but lower Calinski–Harabasz and the fourth cluster still cuts inertia by 16.96%.
> - Reject k = 5: unstable, with least-similar runs at ARI 0.4298.
> - Reject k = 6 or more: less stable and supported only by Davies–Bouldin.
> - Business reading: four rated segments plus the unrated group is a usable number for network reporting.
>
> **Algorithm settings.**
>
> - K-Means: `n_init=10`.
> - K-Means: `k-means++` initialisation.
> - K-Means: `random_state=42`.
> - Hierarchical clustering: Ward linkage.
> - Reason for 10 starts: fifty starts improve inertia by 0.0000% at k = 4 (§2).
> - Reason for Ward linkage: it minimises within-cluster variance, the same quantity K-Means minimises, so the two methods are comparable.

---
## 4. The final K-Means model

K-Means numbers its clusters arbitrarily — cluster 0 is not "first" in any sense, and the numbering can change between
runs. To make segments readable and stable, they are **renumbered in order of their centre's shipment weight**:
segment 1 has the lowest-volume centre. Segment 0 is reserved for the unrated warehouses.

The centres are converted back into recorded units with the scaler saved in the feature engineering step, because a
centre of "+0.8 standard deviations" means nothing to a manager.

In [ ]:
K = 4
kmeans = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(X)

scaler = joblib.load(paths["feature_engine"] / "standard_scaler.pkl")
centres = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=inputs)

# renumber: segment 1 = lowest shipment-weight centre
order = centres["product_wg_ton"].sort_values().index
to_segment = {old: new for new, old in enumerate(order, start=1)}
segment = pd.Series(kmeans.labels_).map(to_segment).to_numpy()

centres = centres.loc[order].reset_index(drop=True)
centres.index = pd.Index(range(1, K + 1), name="segment")
centres.insert(0, "warehouses", pd.Series(segment).value_counts().sort_index().to_numpy())
centres.insert(1, "pct_of_rated", (100 * centres["warehouses"] / len(X)).round(2))
centres.round(2)

> **Interpretation.**
>
> - Translated back into tons and counts, the four centres — each the average warehouse of its
>   segment — describe clearly different kinds of warehouse.
>
> | Segment | Warehouses | Shipment | Refills | Transport issues | Breakdowns | What sets it apart |
> |---|---|---|---|---|---|---|
> | 1 | 6,249 (25.94%) | 12,670 t | 4.15 | 0.36 | 2.07 | **lowest volume, fewest breakdowns** |
> | 2 | 4,018 (16.68%) | 18,093 t | 4.34 | **3.07** | 3.67 | **transport problems** — 8.5 to 11 times the other segments' rate |
> | 3 | 6,172 (25.62%) | 28,349 t | **1.34** | 0.33 | 4.25 | **high volume, rarely refilled** |
> | 4 | 7,653 (31.77%) | 28,870 t | **6.14** | 0.28 | 4.33 | **high volume, frequently refilled** |
>
> - **Segments 3 and 4 differ almost only in refills.** They ship almost the same (28,349 and 28,870 t) and break down
>   almost equally often (4.25 and 4.33), but are refilled 1.34 against 6.14 times. The EDA found refills unrelated
>   to every other measure, so this split is K-Means dividing an independent dimension — a real difference, but the
>   part of the segmentation most likely to depend on the method (§5 tests it).
> - **Breakdowns rise with volume** across segments 1, 3 and 4 (2.07 → 4.25 → 4.33), echoing the shipment–breakdown
>   correlation found in the EDA.
> - **Segment 2 is the only segment defined by a problem** rather than by volume or activity.
>
> - These describe centres only. Whether the segments also differ in age, certification, location or infrastructure is the
>   profiling question for the next step.

---
## 5. Hierarchical clustering (Ward) — does a different algorithm find the same segments?

Hierarchical clustering starts with every warehouse on its own and repeatedly merges the two closest groups. **Ward
linkage** merges whichever pair increases total within-cluster variance least — the same quantity K-Means minimises —
so if both methods divide the warehouses similarly, the segments do not depend on the particular algorithm. Other
linkages (single, complete, average) optimise different quantities and would not be a like-for-like comparison.

**It must run on a sample.** Hierarchical clustering needs the distance between every pair of warehouses held in memory
at once; the cell below prints what that would take for all 24,092. A random sample of 5,000 warehouses is used.

On that sample, the Ward tree is cut into the same number of clusters as K-Means, and the two labellings are compared
with the ARI (§2) and a cross-tabulation.

In [ ]:
n = len(X)
full_pairs = n * (n - 1) // 2
print(f"all {n:,} warehouses: {full_pairs:,} pair distances = {full_pairs * 8 / 1024**3:.2f} GB in memory")

rng = np.random.default_rng(RANDOM_STATE)
sample_idx = np.sort(rng.choice(n, 5_000, replace=False))
sample_pairs = 5_000 * 4_999 // 2
print(f"sample of 5,000        : {sample_pairs:,} pair distances = {sample_pairs * 8 / 1024**2:.0f} MB\n")

X_sample = X[sample_idx]
tree = linkage(X_sample, method="ward")
ward_raw = fcluster(tree, t=K, criterion="maxclust")

# renumber Ward clusters by centre shipment weight, exactly as for K-Means
ward_centre_ship = pd.Series(X_sample[:, 0]).groupby(ward_raw).mean().sort_values()
ward_segment = pd.Series(ward_raw).map({old: new for new, old in enumerate(ward_centre_ship.index, start=1)}).to_numpy()
kmeans_segment_sample = segment[sample_idx]

ari = adjusted_rand_score(kmeans_segment_sample, ward_segment)
print(f"ARI, Ward vs K-Means on the same 5,000 warehouses: {ari:.4f}")
print(f"silhouette on the sample — K-Means: {silhouette_score(X_sample, kmeans_segment_sample):.4f}"
      f"   Ward: {silhouette_score(X_sample, ward_segment):.4f}\n")

pd.crosstab(pd.Series(kmeans_segment_sample, name="K-Means segment"),
            pd.Series(ward_segment, name="Ward segment"), margins=True)

> **Interpretation.**
>
> - The pair distances for all 24,092 warehouses would need 2.16 GB of memory; the 5,000-warehouse
>   sample needs 95 MB — the reason for sampling.
>
> - **Ward and K-Means agree only partly:** ARI **0.3303**, and on the same warehouses Ward's clusters are less well
>   separated (silhouette 0.1603 against K-Means' 0.2372). The cross-tabulation shows *where* they agree:
>
> - **The transport-problem segment is found by both.** Of the 836 sampled warehouses in K-Means segment 2, Ward places
>   721 (86%) together.
> - **The low-volume segment is largely shared.** 938 of the 1,233 in K-Means segment 1 (76%) fall in one Ward cluster.
> - **The two high-volume segments are where the methods part.** K-Means segment 3 splits 527 / 583 across two Ward
>   clusters, and segment 4 splits 977 / 496. Ward draws the boundary through the high-volume warehouses differently
>   from K-Means' refill split.
>
> - This is the sensitivity §4 anticipated.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(tree, truncate_mode="lastp", p=30, show_leaf_counts=True, leaf_rotation=90, ax=ax,
           color_threshold=tree[-(K - 1), 2])
ax.set_title(f"Ward dendrogram — 5,000 sampled warehouses, last 30 merges (coloured at the cut into {K} clusters)",
             fontweight="bold")
ax.set_ylabel("merge distance")
plt.tight_layout()
save_fig(paths["train_eval"] / "ward_dendrogram.png")
plt.show()

heights = tree[-12:, 2][::-1]          # the last 12 merges, largest first
print("Reading the tree: a large gap between two successive merges means the groups below it are far apart,")
print("so cutting the tree inside that gap is a natural place to stop merging.\n")
print(pd.DataFrame({
    "merge_leaves_clusters": range(1, 12),
    "merge_distance": heights[:-1].round(2),
    "gap_below_this_merge": (heights[:-1] - heights[1:]).round(2),
    "a_cut_in_this_gap_gives_k": range(2, 13),
}).to_string(index=False))

> **Interpretation.**
>
> - Read from the dendrogram, cut into four coloured clusters: the merge inside the green cluster
>   (about 57) sits only just below the merge joining the red and purple clusters (about 59), so the cut into four falls in
>   a very narrow band. The table confirms it. The gap a cut into four would occupy is just **1.90**, whereas the largest
>   gap, **18.90**, would give **five** clusters, and the second largest, 14.58, three.
>
> - Taken alone, this tree favours five clusters over four. The next cell checks whether that preference holds beyond this
>   one sample.


**Is that reading of the tree stable?** The tree above comes from one sample of 5,000 warehouses. If a different sample
produced a different natural cut, the tree's preference would be a property of the sample, not of the warehouses. Ward is
therefore rerun on five samples (the first is the one above), recording where each tree's largest gap falls and how far
each tree, cut into the same number of clusters as K-Means, agrees with K-Means on those warehouses.

In [ ]:
rows = []
for s in range(5):
    idx = np.sort(np.random.default_rng(RANDOM_STATE + s).choice(n, 5_000, replace=False))
    tr = linkage(X[idx], method="ward")
    h = tr[-12:, 2][::-1]
    gaps = h[:-1] - h[1:]
    rows.append({
        "sample": s + 1,
        "largest_gap_gives_k": int(np.argmax(gaps) + 2),
        "largest_gap": round(gaps.max(), 2),
        "gap_at_a_cut_into_K": round(gaps[K - 2], 2),
        "ari_ward_vs_kmeans_at_K": round(adjusted_rand_score(segment[idx], fcluster(tr, t=K, criterion="maxclust")), 4),
    })
ward_samples = pd.DataFrame(rows)
ward_samples

> **Interpretation.**
>
> - It does not hold. Across five samples of 5,000 warehouses, **the tree's largest gap points to
>   k = 5, 3, 3, 4 and 4** — its natural cut changes with the sample. In samples 4 and 5 the cut into four *is* the largest
>   gap (18.01 and 16.91); in sample 1 it is only 1.90, against a largest gap of 18.90. A preference that moves from
>   sample to sample belongs to the sample, not to the warehouse network — exactly what the EDA predicts for a
>   continuum, where no cut is clearly natural.
>
> - Agreement with K-Means at four clusters also varies with the sample, from ARI **0.3303 to 0.5461** — moderate at best.
>
> - **What this means for the segment-count decision.** The hierarchical results do not overturn k = 4: they offer no
>   consistent alternative, and in two of five samples they favour four themselves. K-Means remains the reference, because
>   it uses every rated warehouse and reproduces almost perfectly across seeds (ARI 0.9997), whereas each Ward tree sees
>   about a fifth of the data.
>
> - The comparison does set a limit on interpretation, carried to the profiling step: **the low-volume and
>   transport-problem segments hold across methods; the division of high-volume warehouses into frequently and rarely
>   refilled is specific to K-Means.**

---
## 6. Every warehouse gets a segment

The model results are joined back to all 25,000 warehouses:

- **segment 0** — the 908 unrated warehouses, assigned by rule;
- **segments 1 to K** — the 24,092 rated warehouses, assigned by K-Means.

K-Means provides the final labels. It runs on all rated warehouses, whereas Ward could only run on a sample; Ward's role
is to confirm or question the K-Means segments (§5).

In [ ]:
pre = load_preprocessed()

labels = pd.DataFrame({"Ware_house_ID": scaled["Ware_house_ID"], "segment": segment, "assigned_by": "kmeans"})
unrated = pd.DataFrame({"Ware_house_ID": pre.loc[pre["is_unrated_warehouse"] == 1, "Ware_house_ID"],
                        "segment": 0, "assigned_by": "rule: unrated"})
labels = pd.concat([unrated, labels], ignore_index=True)

summary = labels.groupby(["segment", "assigned_by"]).size().rename("warehouses").reset_index()
summary["pct_of_all"] = (100 * summary["warehouses"] / len(labels)).round(2)
summary

> **Interpretation.**
>
> - Every one of the 25,000 warehouses now has exactly one segment: 908 (3.63%) in segment 0 by rule,
>   and 24,092 across segments 1 to 4 by K-Means. Against the whole network, segments 1 to 4 hold 25.00%, 16.07%, 24.69%
>   and 30.61%. The largest is segment 4 (high volume, frequently refilled); the smallest data-driven segment is segment 2
>   (transport problems).


---
## 7. Save

In [ ]:
joblib.dump({"model": kmeans, "segment_of_cluster": to_segment, "inputs": inputs},
            paths["model"] / "kmeans.pkl")
print(f"saved  {(paths['model'] / 'kmeans.pkl').relative_to(PROJECT_ROOT)}")

save_table(scan.reset_index(), paths["train_eval"] / "k_selection.csv", index=False)
save_table(labels, paths["train_eval"] / "segment_labels.csv", index=False)
_ = save_table(pd.DataFrame({"Ware_house_ID": scaled["Ware_house_ID"].to_numpy()[sample_idx],
                         "kmeans_segment": kmeans_segment_sample, "ward_segment": ward_segment}),
           paths["train_eval"] / "ward_vs_kmeans_sample.csv", index=False)

---
## 8. Checks

The saved model is reloaded and asked to label the rated warehouses again; after renumbering it must reproduce the saved
segments exactly. The label file must cover every warehouse once.

In [ ]:
saved = joblib.load(paths["model"] / "kmeans.pkl")
relabelled = pd.Series(saved["model"].predict(X)).map(saved["segment_of_cluster"]).to_numpy()
back = pd.read_csv(paths["train_eval"] / "segment_labels.csv")

assert (relabelled == segment).all(), "reloaded model does not reproduce the segments"
assert len(back) == 25_000 and back["Ware_house_ID"].is_unique
assert set(back["Ware_house_ID"]) == set(pre["Ware_house_ID"])
assert (back["segment"] == 0).sum() == 908
assert set(back.loc[back["assigned_by"] == "kmeans", "segment"]) == set(range(1, K + 1))
assert back.loc[back["segment"] == 0, "Ware_house_ID"].isin(pre.loc[pre["is_unrated_warehouse"] == 1, "Ware_house_ID"]).all()

print("all checks passed")
print(f"segment_labels.csv : {len(back):,} warehouses, each once; segment 0 = 908 unrated; segments 1-{K} by K-Means")
print("kmeans.pkl         : reproduces every K-Means segment")

---
## Summary

**Model.** K-Means with **k = 4** on the four standardised performance measures of 24,092 rated warehouses, plus
**segment 0** — the 908 unrated warehouses — by rule. All 25,000 warehouses have exactly one segment.

| Segment | Share of network | Defining feature (segment centre) |
|---|---|---|
| 0 | 3.63% | unrated, recently commissioned — assigned by rule |
| 1 | 25.00% | lowest volume (12,670 t) and fewest breakdowns (2.07) |
| 2 | 16.07% | transport problems (3.07 a year) |
| 3 | 24.69% | high volume (28,349 t), rarely refilled (1.34) |
| 4 | 30.61% | high volume (28,870 t), frequently refilled (6.14) |

**The segment count and algorithm settings.** k = 4 is supported by Calinski–Harabasz (maximum at 7,661.0), stability
(ARI 0.9997 across seeds), and the sharpest inertia slowdown (16.96% then 11.54%). Davies–Bouldin alone prefers k = 6;
k = 5 is unstable (lowest ARI 0.4298). K-Means uses `n_init=10` (50 starts improve inertia by 0.0000% at k = 4) and
Ward linkage for comparison (minimises the same within-cluster variance as K-Means).

### What the evidence says about these segments

- **Separation is weak, as expected.** Silhouette is 0.22–0.24 at every k: the segments divide a continuum.
- **They are reproducible.** K-Means returns the same four segments from different starts (ARI 0.9997).
- **Part of the result does not depend on the algorithm.** Ward also finds the transport-problem segment (86% of sampled
  K-Means segment 2 kept together) and largely the low-volume segment (76%). The refill split of the high-volume
  warehouses is specific to K-Means (Ward vs K-Means ARI 0.33–0.55 across five samples).
- **Ward's tree has no stable natural cut.** Its largest gap indicates 5, 3, 3, 4 and 4 clusters across five samples.

**Output.** `model/kmeans.pkl` (verified to reproduce every segment) · `training_and_evaluation/segment_labels.csv` ·
`k_selection.csv` · `ward_vs_kmeans_sample.csv` · `k_selection.png` · `ward_dendrogram.png`.

**Handed to the profiling step:** labels for all 25,000 warehouses, the model, and three questions — how well separated
the final segments are (silhouette, Dunn index); whether they differ in the *conditions* Objective 1 asks about (age,
certificate, temperature regulation, location and the rest); and what to call each segment in business language,
bearing in mind that segments 3 and 4 are the least robust to the choice of method.